# 05 Drought and Water Stress
**Series:** Pine Ridge Hydrology                                          
**Author:** Lilly Jones, PhD                                              
**Primary Focus:** Pine Ridge Reservation/Oglala Sioux Tribe                                                      
**Collective:** Oglala Lakota                                                    
**Data Sources:** NOAA PDSI, USGS streamflow (notebook 03)                                                    

## Drought on the Northern Great Plains
Drought is not exceptional on the Pine Ridge study area, it is a recurring
feature of the Northern Great Plains climate. The PDSI record shows
severe drought in 1934–1940 (Dust Bowl), 1956–1957, 1974–1977,
2002–2003, and 2012. These are not rare extremes; multi-year drought
is the expected condition roughly every decade.

What is changing under climate stress is the **severity and duration**
of drought events and the reduced recovery between them. When
groundwater recharge rates are already slow and aquifer levels are
declining, successive drought years have a compounding effect that
single-event analysis misses.

## What PDSI Measures
The Palmer Drought Severity Index (PDSI) accounts for both precipitation
deficit and temperature-driven evapotranspiration demand. Values:
- PDSI ≥ 0: near-normal or wet
- PDSI -1 to -2: mild to moderate drought
- PDSI -3 to -4: severe to extreme drought
- PDSI < -4: exceptional drought

## Research Questions
- Which NOAA climate divisions cover Pine Ridge?
- What is the historical drought frequency for these divisions?
- Do streamflow drought stages (notebook 03) correspond to PDSI?
- Is the frequency of severe drought events increasing?

## Learning Objectives

By the end of this notebook, learners will be able to:

- summarize drought frequency using a regional PDSI proxy
- explain the spatial and conceptual limits of NOAA Climate Division 7
- distinguish correlation from prediction and causation when comparing drought and streamflow

## Prerequisites and Timing

Allow approximately 75–100 minutes. Before beginning, activate the repository environment, read the series governance statement, and complete the preceding notebook where applicable. Work in pairs and rotate analyst, data-steward, skeptic, and documentarian roles.

## Governance Checkpoint

This notebook uses public environmental data describing Oglala Lakota lands and waters. Public availability does not establish permission for every reuse or interpretation. Do not add OST-controlled data, sensitive locations, or community knowledge. Results are educational and screening-level pending OLC/OST review.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml
from scipy import stats

from src.constants import (
    OUTPUTS_DIR, FIGURES_DIR, REPO_ROOT as _REPO_ROOT,
)
from src.config import load_config, streamflow_site_ids, streamflow_site_names

CONFIG = load_config()
STUDY_BBOX = tuple(CONFIG["study_area"]["hydrologic_context_bbox"])
STUDY_NAMES = [CONFIG["study_area"]["people"]]
STUDY_CENTROIDS = {CONFIG["study_area"]["people"]: CONFIG["study_area"]["centroid"]}
PINE_RIDGE_STREAMGAGES = {
    site["name"]: str(site["id"]) for site in CONFIG["usgs_streamflow_sites"]
}

from src.loaders import load_pdsi
from src.indicators import theilsen_trend, normalize_0_1
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

with open(_REPO_ROOT/"config"/"config.yaml") as f:
    CONFIG = yaml.safe_load(f)

DROUGHT_THRESH = CONFIG["thresholds"]["drought"]
START_YEAR     = CONFIG["analysis"]["start_year"]
END_YEAR       = CONFIG["analysis"]["end_year"]

# NOAA Division 7 (Southwest) is the configured regional proxy for Pine Ridge.
# It is not a Reservation-specific observation.
PRIMARY_DIVISIONS = {7: "Pine Ridge (Oglala Lakota)"}

print(f"Primary NOAA climate divisions:")
for div, name in PRIMARY_DIVISIONS.items():
    print(f"  Division {div}: {name}")
print(f"\nDrought thresholds (PDSI):")
for k, v in DROUGHT_THRESH.items():
    print(f"  {k}: {v}")

In [ ]:
# Print data sovereignty statement at the top of every notebook
print_data_acknowledgment(source_keys=["noaa_pdsi", "usgs_nwis_streamflow"])

## Load PDSI Data

In [ ]:
# Load full NOAA South Dakota PDSI record (all 9 climate divisions, 1895–present)

pdsi_all = load_pdsi(
    state_code = "39",   # South Dakota
    start_year = 1895,
    end_year   = END_YEAR,
)

print(f"PDSI records loaded: {len(pdsi_all):,}")
print(f"Divisions: {sorted(pdsi_all['division'].unique().tolist())}")
print(f"Date range: {pdsi_all['date'].min().date()} to {pdsi_all['date'].max().date()}")
print()
print("Division coverage:")
print(
    pdsi_all.groupby(["division", "div_name"])["pdsi"].count()
    .rename("n_months")
    .reset_index()
    .to_string(index=False)
)

In [ ]:
pdsi_primary = pdsi_all[
    (pdsi_all["division"].isin(PRIMARY_DIVISIONS.keys())) &
    (pdsi_all["year"] >= START_YEAR)
].copy()

pdsi_primary["nation"] = pdsi_primary["division"].map(PRIMARY_DIVISIONS)

def classify_pdsi(v):
    if v <= DROUGHT_THRESH["extreme"]:   return "Extreme drought"
    elif v <= DROUGHT_THRESH["severe"]:  return "Severe drought"
    elif v <= DROUGHT_THRESH["moderate"]:return "Moderate drought"
    elif v < 0:                          return "Mild drought"
    else:                                return "Near-normal or wet"

pdsi_primary["drought_class"] = pdsi_primary["pdsi"].apply(classify_pdsi)

print(f"Primary division records: {len(pdsi_primary):,}")
if pdsi_primary.empty:
    print("No records found for configured climate division 7.")
    print(f"Available divisions in dataset: {sorted(pdsi_all['division'].unique())}")
else:
    for div, name in PRIMARY_DIVISIONS.items():
        grp = pdsi_primary[pdsi_primary["division"] == div]
        if grp.empty:
            print(f"Division {div} ({name}): no records found")
            continue
        grp_valid = grp.dropna(subset=["pdsi"])
        if grp_valid.empty:
            print(f"Division {div} ({name}): all PDSI values are NaN")
            continue
        min_idx  = grp_valid["pdsi"].idxmin()
        min_date = grp_valid.loc[min_idx, "date"]
        print(f"Division {div} ({name}):")
        print(f"  Mean PDSI:    {grp_valid['pdsi'].mean():.2f}")
        print(f"  % in drought: {(grp_valid['pdsi'] < -2).mean()*100:.1f}%")
        print(f"  Min (driest): {grp_valid['pdsi'].min():.2f} "
              f"({pd.Timestamp(min_date).strftime('%Y-%m')})")
        print()

In [ ]:
print("Available divisions:", sorted(pdsi_all["division"].unique()))
print("Sample records:")
print(pdsi_all.head(3).to_string(index=False))

In [ ]:
from src.constants import CACHE_DIR

cache_file = CACHE_DIR/"noaa_pdsi_climdiv.txt"
lines = cache_file.read_text().strip().splitlines()

print(f"Total lines: {len(lines)}")
print(f"\nFirst 5 lines (raw):")
for line in lines[:5]:
    parts = line.split()
    print(f"  code='{parts[0]}' len={len(parts[0])}  n_values={len(parts)-1}")
    print(f"  full line: {repr(line[:60])}")

# Check what codes start with 39 (South Dakota)
print(f"\nLines starting with '39':")
sd_lines = [l for l in lines if l.startswith("39")]
print(f"  Count: {len(sd_lines)}")
for line in sd_lines[:5]:
    print(f"  {repr(line[:70])}")

## Drought Frequency and Duration Analysis

In [ ]:
# Annual drought severity metrics
annual_pdsi = (
    pdsi_primary
    .groupby(["division", "nation", "year"])
    .agg(
        mean_pdsi      = ("pdsi", "mean"),
        min_pdsi       = ("pdsi", "min"),
        months_drought = ("pdsi", lambda x: (x < -2).sum()),
        months_severe  = ("pdsi", lambda x: (x <= -3).sum()),
    )
    .reset_index()
)

# Trend in annual drought months
print("TREND IN ANNUAL MONTHS OF MODERATE DROUGHT (PDSI ≤ -2)")
for div, name in PRIMARY_DIVISIONS.items():
    grp = annual_pdsi[annual_pdsi["division"] == div]
    trend = theilsen_trend(grp["months_drought"].values, grp["year"].values)
    print(f"\n  {name}:")
    print(f"    Mean drought months/year: {grp['months_drought'].mean():.1f}")
    print(f"    Trend: {trend['slope_per_decade']:+.2f} months/decade "
          f"({'significant' if trend['significant'] else 'not significant'}, "
          f"p={trend['p_value']:.3f})")

## PDSI–Streamflow Crosswalk

In [ ]:
# Load streamflow annual summary from notebook 03 if available
flow_path = OUTPUTS_DIR/"streamflow_annual_summary.csv"

if flow_path.exists():
    flow_annual = pd.read_csv(flow_path)
    print(f"Streamflow annual summary loaded: {len(flow_annual)} years")

    # Merge with PDSI for Pine Ridge division (7)
    pdsi_pr = annual_pdsi[annual_pdsi["division"] == 7][
        ["year", "mean_pdsi", "months_drought"]
    ]
    merged = flow_annual.merge(pdsi_pr, on="year", how="inner")

    if len(merged) > 5:
        r, p = stats.pearsonr(
            merged["mean_pdsi"].dropna(),
            merged.loc[merged["mean_pdsi"].notna(), "emergency_days"],
        )
        print(f"\nCorrelation: Annual PDSI vs. Emergency-stage streamflow days")
        print(f"  r = {r:.3f}  (p = {p:.3f})")
        print(f"  {'Strong' if abs(r) > 0.6 else 'Moderate' if abs(r) > 0.4 else 'Weak'} "
              f"{'positive' if r > 0 else 'negative'} correlation")
        if r < -0.4:
            print("  Negative r: drier PDSI (lower values) to more emergency-stage days")
else:
    merged = pd.DataFrame()
    print("Streamflow annual summary not found: run notebook 03 first.")
    print("PDSI analysis will continue without the crosswalk.")

## Visualizations

In [ ]:
# PDSI time series for configured climate division 7
DROUGHT_COLORS = {
    "Extreme drought":    "#7B241C",
    "Severe drought":     "#C0392B",
    "Moderate drought":   "#E67E22",
    "Mild drought":       "#F1C40F",
    "Near-normal or wet": "#27AE60",
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, (div, name) in zip(axes, PRIMARY_DIVISIONS.items()):
    grp = pdsi_primary[pdsi_primary["division"] == div].sort_values("date")

    # Fill by drought class
    ax.fill_between(grp["date"], grp["pdsi"],
                    where=grp["pdsi"] < 0, color="#E74C3C", alpha=0.4,
                    label="Drought (PDSI < 0)")
    ax.fill_between(grp["date"], grp["pdsi"],
                    where=grp["pdsi"] >= 0, color="#27AE60", alpha=0.4,
                    label="Normal/Wet (PDSI ≥ 0)")
    ax.plot(grp["date"], grp["pdsi"], color="#2C3E50", linewidth=0.8, alpha=0.7)
    ax.axhline(0, color="black", linewidth=1)
    ax.axhline(DROUGHT_THRESH["moderate"], color="#E67E22",
               linewidth=1, linestyle=":", alpha=0.8)
    ax.axhline(DROUGHT_THRESH["extreme"], color="#7B241C",
               linewidth=1, linestyle=":", alpha=0.8)

    ax.set_ylabel("PDSI", fontsize=9)
    ax.set_title(f"Division {div} {name}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_ylim(-7, 6)
    despine(ax)

plt.suptitle(
    f"Palmer Drought Severity Index for the Oglala Lakota ({START_YEAR}–{END_YEAR})\n"
    f"Dotted lines: Moderate drought (PDSI {DROUGHT_THRESH['moderate']}) | "
    f"Extreme drought (PDSI {DROUGHT_THRESH['extreme']})",
    fontsize=10, fontweight="bold",
)
plt.tight_layout()
try:
    fig.savefig(FIGURES_DIR/"05_pdsi_timeseries.png", dpi=150, bbox_inches="tight")
except Exception:
    pass
plt.show()

In [ ]:
# Annual drought months heatmap
import matplotlib.colors as mcolors

heatmap_data = annual_pdsi.pivot(
    index="nation", columns="year", values="months_drought"
)

if not heatmap_data.empty:
    fig, ax = plt.subplots(figsize=(14, 3))
    im = ax.imshow(
        heatmap_data.values,
        aspect="auto", cmap="YlOrRd",
        vmin=0, vmax=12,
    )
    ax.set_xticks(range(len(heatmap_data.columns)))
    ax.set_xticklabels(
        heatmap_data.columns.tolist(),
        rotation=45, ha="right", fontsize=7,
    )
    ax.set_yticks(range(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index.tolist(), fontsize=9)
    plt.colorbar(im, ax=ax, label="Months of moderate drought (PDSI ≤ -2)")
    ax.set_title(
        "Annual Drought Months\n"
        "Darker = more months in moderate or worse drought",
        fontsize=10, fontweight="bold",
    )
    plt.tight_layout()
    try:
        fig.savefig(FIGURES_DIR/"05_drought_heatmap.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()

In [ ]:
# PDSI vs streamflow crosswalk scatter
if not merged.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    sc = ax.scatter(
        merged["mean_pdsi"], merged["emergency_days"],
        c=merged["year"], cmap="coolwarm_r", s=60,
        alpha=0.8, edgecolors="white",
    )
    plt.colorbar(sc, ax=ax, label="Year")

    for _, row in merged.iterrows():
        if row["emergency_days"] > 100 or row["mean_pdsi"] < -2:
            ax.annotate(
                str(int(row["year"])),
                (row["mean_pdsi"], row["emergency_days"]),
                fontsize=7, xytext=(3, 3), textcoords="offset points",
            )

    # Regression line
    m, b, *_ = stats.linregress(merged["mean_pdsi"], merged["emergency_days"])
    x_range  = np.linspace(merged["mean_pdsi"].min(), merged["mean_pdsi"].max(), 50)
    ax.plot(x_range, m * x_range + b, color="black",
            linewidth=1.5, linestyle="--", alpha=0.6,
            label=f"r = {r:.2f}")

    ax.axvline(DROUGHT_THRESH["moderate"], color="#E67E22",
               linewidth=1, linestyle=":", alpha=0.7,
               label=f"Moderate drought threshold ({DROUGHT_THRESH['moderate']}")
    ax.set_xlabel("Annual mean PDSI", fontsize=10)
    ax.set_ylabel("Emergency-stage streamflow days", fontsize=10)
    ax.set_title(
        "PDSI vs. Streamflow Emergency Days\n"
        "Drier years mean more days below emergency flow threshold",
        fontsize=10, fontweight="bold",
    )
    ax.legend(fontsize=9)
    despine(ax)
    plt.tight_layout()
    try:
        fig.savefig(FIGURES_DIR/"05_pdsi_flow_crosswalk.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()

## Exports

In [ ]:
pdsi_primary.to_csv(OUTPUTS_DIR/"pine_ridge_pdsi_division7.csv", index=False)
annual_pdsi.to_csv(OUTPUTS_DIR/"pdsi_annual_summary.csv", index=False)
print("Exported:")
print("  outputs/pine_ridge_pdsi_division7.csv")
print("  outputs/pdsi_annual_summary.csv")

In [ ]:
print(generate_citations(["noaa_pdsi", "usgs_nwis_streamflow"]))

## Learner Checkpoint

Describe one year with strong apparent agreement and one with disagreement between PDSI and streamflow. List plausible reasons without selecting a cause.

## Interpretation Protocol

Before writing a conclusion, separate:

1. **Observation:** what the computed public data show, including unit, period, spatial scope, and missingness.
2. **Interpretation:** a plausible explanation, stated with uncertainty.
3. **Additional evidence:** literature, local monitoring, expertise, or validation needed to evaluate that explanation.
4. **Decision authority:** who is authorized to approve publication, thresholds, or management action.

Do not convert monitoring absence, association, a screening flag, or scenario output into a causal, regulatory, health, policy, or community conclusion.

## Contribution Activity

Improve one proxy limitation, correlation caveat, period label, or explanation of the PDSI scale. Review the change with a partner and record what became clearer or more defensible.

## Evidence Record and Next Step

Record one regenerated result, its source and scope, one transformation, one limitation, and one question requiring more evidence or local knowledge.

Notebook 06 combines selected historical components into an experimental screening index.